# HW3 — Two retrievers and a model: what RAG actually sends

**Name:**
**Student ID:**
**Group:**
**Repository:**

## AI tool disclosure

State which AI tools you used and for what. Expected and fine; undisclosed use is not.

> > I used Gemini for some explanations, as coding assistant and to help with bugs, especially, environment issues (resolving version conflicts with `transformers` for the `gemma3_text` architecture)

Run this notebook **top to bottom** before you push. A cell with no saved output earns nothing.
The instructions for every section are in `README.md`.

---
# Part A — relevance by hand (1 pt)

| | Text |
|---|---|
| D1 | Large language models can generate text and answer questions. |
| D2 | Retrieval-Augmented Generation retrieves relevant documents before generating an answer. |
| D3 | Embeddings represent text as numerical vectors. |
| Query | How does RAG retrieve information? |

`k1 = 1.5`, `b = 0.75`, `idf(t) = ln((N − n(t) + 0.5) / (n(t) + 0.5))`. Tokenise as README §3 says.

## A1 (a) — BM25 as written

**Tokens of each document, and its length:**

| Doc | Tokens    | \|d\| |
|---|-----------|------:|
| D1 | large, language, models, can, generate, text, and, answer, questions  |     9 |
| D2 | retrieval, augmented, generation, retrieves, relevant, documents, before, generating, an, answer |    10 |
| D3 | embeddings, represent, text, as, numerical, vectors  |     6 |

**avgdl = (9 + 10 + 6) / 3 = 8.333**

**Query terms: how, does, rag, retrieve, information **

**TF, DF and IDF of each query term:**

| Query term  | f(t, D1) | f(t, D2) | f(t, D3) | n(t) (DF) | idf(t) |
|-------------|---------:|---------:|---------:|----------:|-------:|
| how         |        0 |        0 |        0 |         0 |      1.946 |
| does        |        0 |        0 |        0 |         0 |      1.946 |
| rag         |        0 |        0 |        0 |         0 |      1.946 |
| retrieve    |        0 |        0 |        0 |         0 |      1.946 |
| information |        0 |        0 |        0 |         0 |      1.946 |

**BM25 of each document** (write out the sum for every document, one line per term that contributes):

* D1: 0 (no terms match)
* D2: 0 (no terms match, because 'retrieve' != 'retrieval' or 'retrieves')
* D3: 0 (no terms match)

| Document | BM25 Score | Rank |
|---|-----------:|-----:|
| D1 |          0 |    1 |
| D2 |          0 |    1 |
| D3 |          0 |    1 |
* IDF = ln((3 - 0 + 0.5) / (0 + 0.5)) = ln(7) ≈ 1.946

## A1 (b) — BM25 with the stems from README §3

**Stemmed tokens of each document, and its length:**

| Doc | Tokens | \|d\| |
|---|---|---:|
| D1 | large, language, models, can, gener, text, and, answer, questions | 9 |
| D2 | retriev, augmented, gener, retriev, relevant, documents, before, gener, an, answer | 10 |
| D3 | embeddings, represent, text, as, numerical, vectors | 6 |

**avgdl =** 8.333

**Stemmed query terms:** how, doe, rag, retriev, inform

| Query term | f(t, D1) | f(t, D2) | f(t, D3) | n(t) (DF) | idf(t) |
|---|---:|---:|---:|---:|---:|
| how | 0 | 0 | 0 | 0 | 1.946 |
| doe | 0 | 0 | 0 | 0 | 1.946 |
| rag | 0 | 0 | 0 | 0 | 1.946 |
| retriev | 0 | 2 | 0 | 1 | 0.5108 |
| inform | 0 | 0 | 0 | 0 | 1.946 |

**BM25 of each document**, worked:

* D1: Score = 0
* D2 (retriev): 0.5108 * (2 * (1.5 + 1)) / (2 + 1.5 * (1 - 0.75 + 0.75 * (10 / 8.333))) = 2.554 / (2 + 1.5 * 1.15) = 2.554 / 3.725 = 0.6856
* D3: Score = 0

| Document | BM25 Score | Rank |
|---|---:|---:|
| D2 | 0.6856 | 1 |
| D1 | 0 | 2 |
| D3 | 0 | 2 |

**Which document does BM25 retrieve first, in (a) and in (b)? Why?**

> In case (a), BM25 failed to retrieve any document (all scores were 0) because it searched for the exact string 'retrieve', which did not match 'retrieval' or 'retrieves' in D2. In case (b), thanks to stemming, both 'retrieval' and 'retrieves' in D2 were reduced to the stem 'retriev'. Document D2 is retrieved first with a score of ~0.686.

## A2 — cosine similarity

```
Query = [0.8, 0.2, 0.5]
D1 = [0.7, 0.1, 0.4]   D2 = [0.9, 0.3, 0.6]   D3 = [0.1, 0.8, 0.2]
```

| Pair | Dot product (worked) | ‖Query‖ | ‖D‖ | Cosine |
|---|---|---:|---:|---:|
| Query · D1 | (0.8 * 0.7) + (0.2 * 0.1) + (0.5 * 0.4) = 0.56 + 0.02 + 0.20 = 0.78 | 0.9644 | 0.8124 | 0.9956 |
| Query · D2 | (0.8 * 0.9) + (0.2 * 0.3) + (0.5 * 0.6) = 0.72 + 0.06 + 0.30 = 1.08 | 0.9644 | 1.1225 | 0.9977 |
| Query · D3 | (0.8 * 0.1) + (0.2 * 0.8) + (0.5 * 0.2) = 0.08 + 0.16 + 0.10 = 0.34 | 0.9644 | 0.8307 | 0.4244 |

| Document | Cosine Similarity | Rank |
|---|---:|---:|
| D1 | 0.9956 | 2 |
| D2 | 0.9977 | 1 |
| D3 | 0.4244 | 3 |

**Which document is most similar to the query?**

> Document D2 is the most similar with a score of 0.9977, closely followed by D1 (0.9956).

**Is the ranking the same as BM25's?** Compare it with both A1 rankings.

> The ranking partially agrees with BM25 (b) by putting D2 in the first place. But, it differs completely from A1 (a), where all documents scored 0. In A1 (b), D1 and D3 were tied with a score of 0. Here, the embedding model successfully differentiates them, giving D1 high similarity score despite zero keyword overlap.

**Why can BM25 and embedding retrieval disagree?** Point at the words of this query and these documents.

> BM25 requires exact lexical overlap (or stem overlap). For D1, BM25 gave a score of 0 because the query words "RAG", "retrieve", and "information" dont appear in D1's text at all. Embeddings, capture semantic meaning. D1 states: "Large language models can generate text and answer questions", and it describes what RAG does. The embedding model recognizes deep semantic relationship and scores D1 highly (~0.995), while BM25 fails because the specific words don't match.

Optional: check your Part A arithmetic here. The working above must still be written out by hand.

In [5]:
import math

q = [0.8, 0.2, 0.5]
d1 = [0.7, 0.1, 0.4]
d2 = [0.9, 0.3, 0.6]
d3 = [0.1, 0.8, 0.2]

def dot(a, b): return sum(x*y for x, y in zip(a, b))
def mag(a): return math.sqrt(sum(x*x for x in a))
def cos_sim(a, b): return dot(a, b) / (mag(a) * mag(b))

print("Cosine similarity")
print(f"D1: {cos_sim(q, d1):.4f}")
print(f"D2: {cos_sim(q, d2):.4f}")
print(f"D3: {cos_sim(q, d3):.4f}")

N = 3
n_t = 1
idf = math.log((N - n_t + 0.5) / (n_t + 0.5))

f_td = 2
k1 = 1.5
b = 0.75
dl = 10
avgdl = 8.333

bm25_score = idf * (f_td * (k1 + 1)) / (f_td + k1 * (1 - b + b * (dl / avgdl)))
print("\nBM25 score")
print(f"D2: {bm25_score:.4f}")

Cosine similarity
D1: 0.9956
D2: 0.9977
D3: 0.4244

BM25 score
D2: 0.6857


---
# Part B — the same two retrievers, from libraries (1.5 pt)

Load `data/corpus.jsonl` and `data/queries.json`.

In [6]:
import json
import numpy as np

corpus = []
with open('data/corpus.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        corpus.append(json.loads(line))

with open('data/queries.json', 'r', encoding='utf-8') as f:
    queries = json.load(f)['queries']

print(f"Docs loaded: {len(corpus)}")
print(f"Requests loaded: {len(queries)}")

Docs loaded: 74
Requests loaded: 8


## B1 — BM25 with `rank_bm25`

**My tokenisation:** (lowercase? regex? stemming? stop words?)

>I convert all text to lowercase and extract words using the regex \w+. I dont use stemming or remove stop words for this library implementation.

In [7]:
import re
from rank_bm25 import BM25Okapi

def tokenize(text):
    return re.findall(r'\w+', text.lower())

doc_texts = [doc['text'] for doc in corpus]
tokenized_corpus = [tokenize(text) for text in doc_texts]

bm25 = BM25Okapi(tokenized_corpus, k1=1.5, b=0.75)

print("BM25 created")
#top-3 will be later

BM25 created


l## B2 — EmbeddingGemma (`google/embeddinggemma-300m`) with `encode_document` / `encode_query`

In [8]:
import os
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer

load_dotenv()
hf_token = os.getenv("HF_TOKEN")

print("Model load")
model = SentenceTransformer("google/embeddinggemma-300m", token=hf_token)

print("Embedding")
doc_embeddings = model.encode_document(doc_texts)

print("Done. Form:", doc_embeddings.shape)

Model load


Loading weights: 100%|██████████| 314/314 [00:00<00:00, 1809.79it/s]


Embedding
Done. Form: (74, 768)


## My own queries

Write them, and their `relevant` chunk ids, **before** you run them. Say which one you expect BM25 to win and which you expect embeddings to win, and why.

| Id | Query | `relevant` | Expected winner, and why |
|---|---|---|---|
| Q-09 | What are the rules of the archived 2025 scheme? | ["GO-05"] | BM25. The query contains highly specific keywords ("archived", "2025", "scheme") that perfectly match the text in GO-05. |
| Q-10 | What alcoholic beverage is produced from mare's milk? | ["KZ-39", "KZ-40"] | Embeddings. The text describes kumis as a "fermented dairy product", while the query asks for an "alcoholic beverage". Embeddings will understand the semantic equivalence, whereas BM25 will likely fail due to lack of exact word overlap. |

In [9]:
my_queries = [
    {"id": "Q-09", "text": "What are the rules of the archived 2025 scheme?", "relevant": ["GO-05"]},
    {"id": "Q-10", "text": "What alcoholic beverage is produced from mare's milk?", "relevant": ["KZ-39", "KZ-40"]}
]

if len(queries) == 8:
    queries.extend(my_queries)

print(f"Overall requests for check: {len(queries)}")

Overall requests for check: 10


## Results for all ten or more queries: top 3 from each retriever

In [11]:
def get_hit_at_3(retrieved_ids, relevant_ids):
    return any(doc_id in relevant_ids for doc_id in retrieved_ids[:3])

bm25_hits = 0
embed_hits = 0

for q in queries:
    print(f"\nRequest {q['id']}: {q['text']}")
    print(f"Expected relevant: {q['relevant']}")

    # BM25
    q_tokens = tokenize(q['text'])
    bm25_scores = bm25.get_scores(q_tokens)
    bm25_top3_idx = np.argsort(bm25_scores)[::-1][:3]
    bm25_top3_ids = [corpus[i]['id'] for i in bm25_top3_idx]

    hit_bm25 = get_hit_at_3(bm25_top3_ids, q['relevant'])
    if hit_bm25: bm25_hits += 1

    print("\n[BM25 Top-3]")
    for rank, idx in enumerate(bm25_top3_idx):
        doc = corpus[idx]
        print(f"{rank+1}. {doc['id']} (Score: {bm25_scores[idx]:.4f}) - {doc['text'][:80]}...")
    print(f"-> BM25 Hit@3: {hit_bm25}")

    # EmbeddingGemma
    q_embed = model.encode_query(q['text'])
    # Cos similarity
    similarities = model.similarity(q_embed, doc_embeddings)[0].numpy()
    embed_top3_idx = np.argsort(similarities)[::-1][:3]
    embed_top3_ids = [corpus[i]['id'] for i in embed_top3_idx]

    hit_embed = get_hit_at_3(embed_top3_ids, q['relevant'])
    if hit_embed: embed_hits += 1

    print("\n[EmbeddingGemma Top-3]")
    for rank, idx in enumerate(embed_top3_idx):
        doc = corpus[idx]
        print(f"{rank+1}. {doc['id']} (Score: {similarities[idx]:.4f}) - {doc['text'][:80]}...")
    print(f"-> EmbeddingGemma Hit@3: {hit_embed}")
    print("-" * 60)

print(f"\nRESULTS Hit@3:")
print(f"BM25: {bm25_hits}/10")
print(f"EmbeddingGemma: {embed_hits}/10")


Request Q-01: What is form GR-305 used for?
Expected relevant: ['GO-08', 'GO-12', 'GO-23']

[BM25 Top-3]
1. GO-08 (Score: 15.8255) - Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal f...
2. GO-12 (Score: 8.1953) - A student who disagrees with a decision may appeal within 14 calendar days of th...
3. GO-06 (Score: 8.0700) - Form GR-104 is the application form for the need-based study grant. It is the on...
-> BM25 Hit@3: True

[EmbeddingGemma Top-3]
1. GO-08 (Score: 0.6955) - Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal f...
2. GO-12 (Score: 0.4609) - A student who disagrees with a decision may appeal within 14 calendar days of th...
3. GO-23 (Score: 0.4505) - Если студент не согласен с решением, он может подать апелляцию по форме GR-305 в...
-> EmbeddingGemma Hit@3: True
------------------------------------------------------------

Request Q-02: My family is poor. Is there financial help for students like me?
Expected rel

## B3 — comparison

| Query | BM25 top 3 (ids)    | BM25 hit@3 | Embedding top 3 (ids) | Embedding hit@3 |
|---|---------------------|:----------:|-----------------------|:---:|
| Q-01 | GO-08, GO-12, GO-06 |    true    | GO-08, GO-12, GO-23   |true |
| Q-02 | KZ-17, GO-20, KZ-25 |   false    | GO-01, GO-10, GO-20   |true |
| Q-03 | GO-14, GO-23, GO-22 |    true    | GO-14, GO-09, GO-11   |true |
| Q-04 | GO-05, GO-17, GO-19 |   false    | GO-02, GO-05, GO-17   |true |
| Q-05 | KZ-05, KZ-06, KZ-11 |    true    | KZ-05, KZ-06, KZ-03   |true |
| Q-06 | KZ-37, KZ-38, KZ-03 |    true    | KZ-37, KZ-38, KZ-10   |true |
| Q-07 | GO-21, GO-16, GO-01 |   false    | GO-15, GO-18, GO-01   |true |
| Q-08 | GO-17, GO-05, GO-04 |    true    | GO-17, GO-18, GO-15   |true |
| Q-09 | GO-05, GO-20, GO-13 |    true    | GO-05, GO-02, GO-22   |true |
| Q-10 | KZ-39, KZ-40, KZ-49 |    true    | KZ-40, KZ-39, KZ-49   |true |
| **hit@3 total** | 7                   |    /10     | 10                    | /10 |

**Both retrievers agree:** which query, and why was it easy?

> Q-01. It was easy because the query contained specific, rare keyword ("GR-305") which BM25 matched perfectly by exact token, while the embedding model successfully captured the semantic intent of asking for the form's purpose.

**BM25 did better:** which query, and what exact string did it match that the embedding model blurred?

> Q-09. BM25 matched the exact string "archived", "2025", and "scheme" in chunk GO-05. Embedding models can sometimes blur specific numerical or archival distinctions in favor of broader semantic similarities about grants, making BM25 good for exact keyword lookups.

**Embeddings did better:** which query, and what could BM25 not match?

> Q-07. Embeddings correctly retrieved GO-15, completely bypassing the language barrier. Model understood that the English query about a "mother's card" is semantically equivalent to the Russian text about "перевод на счёт родителей" (transfer to parents account). BM25 failed completely because it only looked for exact English word overlaps.

---
# Part C — connect retrieval to the model (1.5 pt)

Model: `gpt-5.6-luna` for every call. Top 3 unless stated.

In [12]:
import json
import os
import tiktoken
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

enc = tiktoken.get_encoding("o200k_base")

with open('data/qa.json', 'r', encoding='utf-8') as f:
    qa_data = json.load(f)['questions']

print(f"Q loaded: {len(qa_data)}")

Q loaded: 6


## C1 — the complete prompt, printed before it is sent

In [14]:
# Q1
q1 = qa_data[0]
question_text = q1['question']

# EG t3
q_embed = model.encode_query(question_text)
similarities = model.similarity(q_embed, doc_embeddings)[0].numpy()
top3_idx = np.argsort(similarities)[::-1][:3]
top3_docs = [corpus[i] for i in top3_idx]

system_msg = (
    "Answer the question using only the provided context.\n"
    "If the answer cannot be found in the context, say that you do not know.\n"
    "Cite the context blocks you used, like [1] or [2][3]."
)

context_blocks = ""
for i, doc in enumerate(top3_docs):
    context_blocks += f"[{i+1}] {doc['text']}\n"

prompt_content = f"Context:\n{context_blocks}\nQuestion: {question_text}"

print("FULL PROMPT BEFORE SENDING")
print("SYSTEM")
print(system_msg)
print("USER")
print(prompt_content)
print("===\n")

# tiktok local count
sys_tokens = len(enc.encode(system_msg))
q_tokens = len(enc.encode(question_text))
ctx_tokens = len(enc.encode(context_blocks))
total_manual = sys_tokens + q_tokens + ctx_tokens

print(f"[Tiktoken counts]")
print(f"System instructions: {sys_tokens}")
print(f"Question: {q_tokens}")
print(f"Retrieved context: {ctx_tokens}")
print(f"Total input (manual): {total_manual}")

print(f"\n[Context Stats]")
print(f"Number of retrieved chunks: {len(top3_docs)}")
for i, doc in enumerate(top3_docs):
    print(f"Chunk {i+1} tokens: {len(enc.encode(doc['text']))}")
print(f"Total context size (chars): {len(context_blocks)}")

print("\nSending req to LLM")
response = client.chat.completions.create(
    model="gpt-5.6-luna",
    messages=[
        {"role": "system", "content": system_msg},
        {"role": "user", "content": prompt_content}
    ],
)

#compare
print(f"usage.prompt_tokens: {response.usage.prompt_tokens}")
print("\n[LLM Answer]")
print(response.choices[0].message.content)

FULL PROMPT BEFORE SENDING
SYSTEM
Answer the question using only the provided context.
If the answer cannot be found in the context, say that you do not know.
Cite the context blocks you used, like [1] or [2][3].
USER
Context:
[1] The grant amount depends only on the income band, not on the GPA. A student in band 1 receives 250,000 tenge for the academic year. A student in band 2 receives 150,000 tenge for the academic year. The amount is paid in two equal instalments, the first in October and the second in February. There is no partial grant: a student either receives the full amount for their band or nothing.
[2] The income band is set from the monthly household income per person, declared on form GR-210. Band 1: below 85,000 tenge per person per month. Band 2: from 85,000 up to 150,000 tenge per person per month. Band 3: above 150,000 tenge per person per month. Households in band 3 are not eligible for the need-based grant, although their students may still apply for merit scholars

| Part of the input | Tokens (tiktoken `o200k_base`) |
|---|-------------------------------:|
| System instructions |                             44 |
| Question |                             18 |
| Retrieved context |                            288 |
| **Total input** |                            350 |
| **`usage.prompt_tokens` from the response** |                            364 |

- Number of retrieved chunks: 3
- Tokens per chunk: ch1 90, ch2 104, ch3 85.
- Total context size (tokens / characters): 288 / 1144

**Why do the tiktoken total and `usage.prompt_tokens` differ?**
> The tiktoken count measures only the raw text.

**What exactly does retrieval add to the model's input?**
> Retrieval literally injects the raw text strings of the highest-scoring chunks directly into the user's prompt alongside the question. The model has no internal "search mechanism" during generation, so it simply reads the inserted text blocks as part of the conversation history and generates the next words.

## C2 — three systems, six questions

In [16]:
def get_llm_answer(question, context_docs=None):
    if context_docs:
        system_msg = (
            "Answer the question using only the provided context.\n"
            "If the answer cannot be found in the context, say that you do not know.\n"
            "Cite the context blocks you used, like [1] or [2][3]."
        )
        context_blocks = ""
        for i, doc in enumerate(context_docs):
            context_blocks += f"[{i+1}] {doc['text']}\n"
        prompt_content = f"Context:\n{context_blocks}\nQuestion: {question}"
    else:
        system_msg = "Answer the question directly."
        prompt_content = question

    response = client.chat.completions.create(
        model="gpt-5.6-luna",
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user", "content": prompt_content}
        ],
    )
    return response.choices[0].message.content

for q in qa_data:
    print(f"\n{q['id']}: {q['question']}")
    print(f"Reference: {q['reference']}\n")

    # llm only (no context)
    print("[LLM only]")
    print(get_llm_answer(q['question']))

    # BM25 RAG
    q_tokens = tokenize(q['question'])
    bm25_scores = bm25.get_scores(q_tokens)
    bm25_top3_idx = np.argsort(bm25_scores)[::-1][:3]
    bm25_docs = [corpus[i] for i in bm25_top3_idx]
    print("\n[BM25 RAG]")
    print(get_llm_answer(q['question'], bm25_docs))

    # Embedding RAG
    q_embed = model.encode_query(q['question'])
    similarities = model.similarity(q_embed, doc_embeddings)[0].numpy()
    embed_top3_idx = np.argsort(similarities)[::-1][:3]
    embed_docs = [corpus[i] for i in embed_top3_idx]
    print("\n[Embedding RAG]")
    print(get_llm_answer(q['question'], embed_docs))
    print("-" * 60)


C-01: How much does a student in income band 2 receive, and when is it paid?
Reference: 150,000 tenge for the academic year, in two equal instalments: October and February.

[LLM only]
Which student support scheme or institution are you referring to? “Income band 2” varies by programme, so I need the scheme name or a link/document to identify the payment amount and schedule.

[BM25 RAG]
A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments: 75,000 tenge in October and 75,000 tenge in February**. [1]

[Embedding RAG]
A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments of 75,000 tenge**: one in **October** and one in **February**. [1]
------------------------------------------------------------

C-02: My grant application was refused. How long do I have to appeal, and which form do I use?
Reference: 14 calendar days from the date on the decision letter, using form GR-305.

[LL

Mark each answer ✓ (agrees with `reference`) or , and give a few words of the answer.

| Question | LLM only                          | BM25 RAG                          | Embedding RAG                     |
|---|-----------------------------------|-----------------------------------|-----------------------------------|
| C-01 | ✗ (Asked for scheme name)     | ✓ (150,000, two instalments)   | ✓ (150,000, two instalments)      |
| C-02 | ✗ (Asked for program details) | ✓ (14 days, GR-305)               | ✓ (14 days, GR-305)               |
| C-03 | ✗ (Asked for grant name)      | ✓ (2.67 on 4.0 scale)             | ✓ (2.67 on 4.0 scale)             |
| C-04 | ✓ (Oct 2017, 2031)             | ✓ (Oct 2017, 2031)                | ✓ (Oct 2017, 2031)                |
| C-05 | ✓ (1963, Alma-Ata Institute)   | ✓ (1963, Almaty Institute)        | ✓ (1963, Almaty Institute)        |
| C-06 | ✗ (Guessed US Pell Grants)    | ✓ (Doesn't know, separate scheme) | ✓ (Doesn't know, separate scheme) |
| **agrees with `reference`** | 2/6                               | 6/6                               | 6/6                               |

## C3 — question 1 of 3: (id)

**1. What BM25 retrieved** (top 3, ids and scores):

> GO-04 (20.2192), GO-02 (18.7422), GO-01 (17.2430)

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

> GO-04 (0.5845), GO-03 (0.4842), GO-15 (0.4506)

**3. What was sent to the model** (the context blocks):

> The raw text of the top 3 chunks for each run. Both models successfully included GO-04

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

> Without context, the LLM refused to answer, asking for the specific grant scheme name. With the retrieved context, both RAG pipelines became highly specific and grounded, providing the exact accurate answer: 150,000 tenge paid in October and February

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

> Nothing was missed. The generation was perfect because the retrieval successfully found the exact chunk (GO-04) containing both the exact amount and the payment schedule

In [18]:
q1 = next(q for q in qa_data if q['id'] == 'C-01')
print(f"Question: {q1['question']}\n")

print("[BM25 Top-3]")
q1_tokens = tokenize(q1['question'])
scores_bm25 = bm25.get_scores(q1_tokens)
for idx in np.argsort(scores_bm25)[::-1][:3]:
    print(f"{corpus[idx]['id']} (Score: {scores_bm25[idx]:.4f})")

print("\n[EmbeddingGemma Top-3]")
q1_embed = model.encode_query(q1['question'])
scores_embed = model.similarity(q1_embed, doc_embeddings)[0].numpy()
for idx in np.argsort(scores_embed)[::-1][:3]:
    print(f"{corpus[idx]['id']} (Score: {scores_embed[idx]:.4f})")

Question: How much does a student in income band 2 receive, and when is it paid?

[BM25 Top-3]
GO-04 (Score: 20.2192)
GO-02 (Score: 18.7422)
GO-01 (Score: 17.2430)

[EmbeddingGemma Top-3]
GO-04 (Score: 0.5845)
GO-03 (Score: 0.4842)
GO-15 (Score: 0.4506)


## C3 — question 2 of 3: (id)

**1. What BM25 retrieved** (top 3, ids and scores):

> GO-16 (15.7985), GO-02 (14.0587), GO-06 (13.8516)

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

> GO-02 (0.6286), GO-20 (0.5691), GO-05 (0.5451)

**3. What was sent to the model** (the context blocks):

> The text of the respective top 3 chunks. Both retrievers successfully included GO-02

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

> Without retrieval, the LLM refused to answer ("Which need-based grant do you mean? The minimum GPA varies..."). With retrieval, the model generated a concise, grounded answer based purely on the facts: "2.67 on a 4.0 scale"

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

> Nothing was missed. It was a flawless generation because the retrieval successfully placed the correct factual chunk (GO-02) into the context window, eliminating the model's uncertainty

In [19]:
q3 = next(q for q in qa_data if q['id'] == 'C-03')
print(f"Question: {q3['question']}\n")

print("[BM25 Top-3]")
q3_tokens = tokenize(q3['question'])
scores_bm25_3 = bm25.get_scores(q3_tokens)
for idx in np.argsort(scores_bm25_3)[::-1][:3]:
    print(f"{corpus[idx]['id']} (Score: {scores_bm25_3[idx]:.4f})")

print("\n[EmbeddingGemma Top-3]")
q3_embed = model.encode_query(q3['question'])
scores_embed_3 = model.similarity(q3_embed, doc_embeddings)[0].numpy()
for idx in np.argsort(scores_embed_3)[::-1][:3]:
    print(f"{corpus[idx]['id']} (Score: {scores_embed_3[idx]:.4f})")

Question: What is the minimum GPA needed to qualify for the need-based grant?

[BM25 Top-3]
GO-16 (Score: 15.7985)
GO-02 (Score: 14.0587)
GO-06 (Score: 13.8516)

[EmbeddingGemma Top-3]
GO-02 (Score: 0.6286)
GO-20 (Score: 0.5691)
GO-05 (Score: 0.5451)


## C3 — question 3 of 3: (id)

**1. What BM25 retrieved** (top 3, ids and scores):

> GO-16 (16.9958), GO-01 (14.1725), GO-20 (13.7202)

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

> GO-20 (0.6664), GO-16 (0.6161), GO-01 (0.5539)

**3. What was sent to the model** (the context blocks):

> The chunks explaining that the grant is for undergrads (GO-01) and that master's students use a separate scheme (GO-20)

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

> Without context, the model completely hallucinated, outputting irrelevant information about US Federal Pell Grants and FAFSA. With context, the model correctly stated that it does not know the exact amount because master's students have a separate scheme

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

> The model missed a specific monetary amount, but this was a deliberate feature of the corpus, not an error. Neither retrieval nor generation failed; the RAG pipeline perfectly executed the system instruction: "If the answer cannot be found in the context, say that you do not know."

In [20]:
q6 = next(q for q in qa_data if q['id'] == 'C-06')
print(f"Question: {q6['question']}\n")

print("[BM25 Top-3]")
q6_tokens = tokenize(q6['question'])
scores_bm25_6 = bm25.get_scores(q6_tokens)
for idx in np.argsort(scores_bm25_6)[::-1][:3]:
    print(f"{corpus[idx]['id']} (Score: {scores_bm25_6[idx]:.4f})")

print("\n[EmbeddingGemma Top-3]")
q6_embed = model.encode_query(q6['question'])
scores_embed_6 = model.similarity(q6_embed, doc_embeddings)[0].numpy()
for idx in np.argsort(scores_embed_6)[::-1][:3]:
    print(f"{corpus[idx]['id']} (Score: {scores_embed_6[idx]:.4f})")

Question: I am a master's student. How much need-based grant can I get?

[BM25 Top-3]
GO-16 (Score: 16.9958)
GO-01 (Score: 14.1725)
GO-20 (Score: 13.7202)

[EmbeddingGemma Top-3]
GO-20 (Score: 0.6664)
GO-16 (Score: 0.6161)
GO-01 (Score: 0.5539)


---
# Final questions

**1. What is the main difference between BM25 and embedding retrieval?**

> BM25 is a sparse, lexical retrieval algorithm that relies on exact keyword matching and term frequency. Embedding retrieval uses dense neural vectors to capture the underlying semantic meaning and context of the text, matching concepts rather than just specific words.

**2. When can BM25 perform better than embeddings?**

> BM25 excels when the user searches for highly specific terminology, exact IDs (like "GR-305"), serial numbers, names, or rare acronyms. In these cases, embeddings might over-generalize or "blur" the specific constraint in favor of broader semantic themes.

**3. When can embeddings perform better than BM25?**

> Embeddings are superior when the query uses synonyms, paraphrasing, or different phrasing than the source text (e.g., matching "alcoholic beverage" to "fermented dairy product"). They also seamlessly handle cross-lingual queries, finding Russian or Kazakh text from an English prompt, which is impossible for standard BM25 without exact translation.

**4. Why is cosine similarity used with embeddings?**

> Cosine similarity measures the angle between two multi-dimensional vectors. It evaluates how closely their semantic directions align, completely ignoring the magnitude (which often correlates with document length). This makes it perfect for comparing a short query vector to a much longer document vector.

**5. What information is actually sent to an LLM in a RAG system?**

> A single compiled text string is sent. It contains the system instructions, the raw text blocks of the retrieved chunks (the context), and the user's prompt.

**6. Does the LLM search the original documents itself?**

> No. The LLM has no active search mechanism. It only reads the specific text injected into its context window by the retrieval pipeline and generates a response based solely on that injected text.

**7. What happens if retrieval returns an irrelevant chunk?**

> If the retrieved context is irrelevant, the LLM might become confused and hallucinate a wrong answer by trying to force connections. However, if properly constrained by a strict system prompt (like ours), it will successfully recognize the irrelevance and state that it does not know the answer.

**8. Why can increasing Top-K sometimes improve RAG and sometimes make it worse? Report K = 1, 3 and 8 on one question.**

> Increasing K improves RAG by raising the probability of catching the correct chunk if it was ranked slightly lower. However, it can make RAG worse by injecting too much irrelevant "noise" that distracts the LLM, and it consumes more API tokens (increasing cost and potentially hitting context window limits).
> **Results for C-02 (Embeddings):**
> K=1: ['GO-08'] (Captured the main form ID)
> K=3: ['GO-08', 'GO-12', 'GO-11'] (Captured appeal rules in English but missed the Russian version, and added noise about missing documents in GO-11)
> K=8: ['GO-08', 'GO-12', 'GO-11', 'GO-23', 'GO-06', 'GO-09', 'GO-17', 'GO-21'] (Successfully caught the Russian version GO-23, but injected a lot of unnecessary noise about other forms and deadlines)

In [21]:
q_test = next(q for q in qa_data if q['id'] == 'C-02')
print(f"Question: {q_test['question']}")
print(f"Expected: {q_test['relevant']}\n")

q_embed = model.encode_query(q_test['question'])
sims = model.similarity(q_embed, doc_embeddings)[0].numpy()

for k in [1, 3, 8]:
    top_k_idx = np.argsort(sims)[::-1][:k]
    retrieved = [corpus[i]['id'] for i in top_k_idx]
    print(f"Top-{k}: {retrieved}")

Question: My grant application was refused. How long do I have to appeal, and which form do I use?
Expected: ['GO-08', 'GO-12', 'GO-23']

Top-1: ['GO-08']
Top-3: ['GO-08', 'GO-12', 'GO-11']
Top-8: ['GO-08', 'GO-12', 'GO-11', 'GO-23', 'GO-06', 'GO-09', 'GO-17', 'GO-21']


**9. Is a vector database required to build a RAG system?**

> No. For small-scale datasets (like our corpus of 74 chunks), simple in-memory array operations using libraries like NumPy (to calculate cosine similarities and sort arrays) are extremely fast and sufficient. Vector databases are only strictly necessary when scaling to millions of documents to ensure efficient indexing and retrieval speed.

**10. Which retrieval approach worked better for this corpus, and what evidence supports that? Give hit@3 for both over your ten queries and the C2 agreement counts for both RAG systems.**

> The dense embedding approach (EmbeddingGemma) worked noticeably better.
> **Evidence:** Over the 10 custom and standard queries, Embeddings achieved a perfect Hit@3 of 10/10, whereas BM25 achieved only 7/10. While both systems tied in the C2 generation task (both scoring 6/6 agreement with the reference), the embedding model demonstrated far superior semantic understanding, correctly matching cross-lingual queries and resolving synonyms that caused BM25 to fail entirely.

---
# Conclusion

> Building this RAG pipeline demonstrated that an LLM's accuracy is fundamentally bounded by the quality of its retrieval system. Without grounding, the model hallucinates wildly; with accurate retrieval and strict system prompting, it provides highly precise answers. While BM25 remains a powerful and lightweight tool for exact keyword lookups, dense embeddings are critical for building intelligent systems capable of understanding synonyms, paraphrasing, and cross-lingual intent.